<a href="https://colab.research.google.com/github/traderjohnd/foundation-model-from-scratch/blob/main/notebooks/02_tokenizer_training_and_corpus_construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a Foundation Model from Scratch
## Notebook 02 — Tokenizer Training & Corpus Construction

This notebook begins from the verified data contract established in **Notebook 01 — Data Preparation & Corpus Audit**. It independently reloads the immutable WikiText-103 revision, applies the same locked normalization and article-reconstruction logic through `src/data.py`, trains the project tokenizer from scratch, validates it, records its checksum, and constructs the exact 20,000,000-token model-training corpus.

Notebook 01 is a completed audit artifact. Notebook 02 must not depend on Notebook 01's in-memory state.

### Pipeline boundary

```text
Notebook 01: raw WikiText → verified normalized articles
src/data.py: canonical reusable loading/normalization/reconstruction logic
Notebook 02: normalized articles → tokenizer → exact 20M-token corpus
Notebook 03: tokenizer/corpus → Transformer architecture
```


### Locked inputs and constraints

- Dataset: `Salesforce/wikitext`, `wikitext-103-raw-v1`
- Immutable Hub revision: `b08601e04326c79dfdd32d625aee71d232d685c3`
- Tokenizer: byte-level BPE trained from scratch
- Vocabulary size: 16,384 total tokens, including registered special tokens
- Tokenizer-training text: full normalized official training split only
- Model-training corpus: exactly 20,000,000 tokenizer-produced tokens
- Sampling seed: 42
- Validation remains development-visible; test remains untouched until final evaluation
- Article boundaries and normalization must reproduce Notebook 01's verified 28,472 training documents and 60 validation documents before tokenizer work proceeds

Canonical references: `docs/PROJECT_CONTEXT.md` and `docs/DECISION_REGISTER.md`.

# Chunk 1 — Reproduce the audited corpus from shared source code

Before tokenizer design begins, this chunk proves that Notebook 02 can independently reproduce Notebook 01's audited corpus. The verified normalization and article-reconstruction implementation has been extracted into `src/data.py` without refactoring the core logic.

The notebook pins the **source-code revision** containing that extraction. This matters because the Hub dataset revision alone fixes the upstream text, while the source-code revision fixes the exact transformation applied to it.

## 1. Load the canonical data pipeline at a fixed source revision

A Colab notebook opened from GitHub does not automatically make the repository's `src/` package importable. We therefore clone the project repository, check out the exact commit that introduced the verified extraction, and add the repository root to Python's import path.

This is intentionally a source-code pin, not a dependency install. The goal is for a fresh runtime to reconstruct the same data pipeline without relying on Notebook 01 or on whatever happens to be at the tip of `main` later.

In [ ]:
%pip install -q datasets


In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/traderjohnd/foundation-model-from-scratch.git"
REPO_DIR = Path("/content/foundation-model-from-scratch")
DATA_PIPELINE_REVISION = "7d300f14c812d9a1caf36aa9ec0568bee5b0f275"

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", DATA_PIPELINE_REVISION], check=True)

sys.path.insert(0, str(REPO_DIR))
print(f"Loaded project source revision: {DATA_PIPELINE_REVISION}")


In [ ]:
import pandas as pd

from src.data import (
    AUDIT_SPLITS,
    DATASET_CONFIG,
    DATASET_ID,
    DATASET_REVISION,
    EXPECTED_ARTICLE_COUNTS,
    load_pinned_wikitext,
    normalize_development_splits,
    reconstruct_development_articles,
    run_normalization_self_test,
)

pd.Series({
    "dataset": DATASET_ID,
    "config": DATASET_CONFIG,
    "dataset_revision": DATASET_REVISION,
    "source_revision": DATA_PIPELINE_REVISION,
    "development_splits": AUDIT_SPLITS,
})


## 2. Re-run the locked normalization regression tests

Notebook 01 established 17 explicit normalization cases. Running the same cases through `src/data.py` checks that extraction did not silently change the transformation contract.

In [ ]:
normalization_test_results = run_normalization_self_test()
normalization_tests = pd.DataFrame(normalization_test_results)

assert len(normalization_tests) == 17
assert normalization_tests["passed"].all()
print("✓ 17/17 normalization regression tests passed.")
normalization_tests


## 3. Reload the immutable WikiText-103 revision

The module loads the same pinned Hub commit used by Notebook 01 and hard-checks the official row counts. Loading the dataset does **not** authorize development-time inspection of test examples; only `train` and `validation` are transformed below.

In [ ]:
raw_dataset = load_pinned_wikitext()

split_rows = pd.Series(
    {split_name: raw_dataset[split_name].num_rows for split_name in raw_dataset},
    name="rows",
)
split_rows


## 4. Normalize development-visible splits only

The same locked function is applied to the official training and validation splits. The test split is deliberately not normalized or inspected during development.

In [ ]:
normalized_development = normalize_development_splits(raw_dataset)

assert set(normalized_development) == {"train", "validation"}
print("✓ Normalized only train and validation.")


## 5. Reconstruct articles and enforce the audit contract

Article starts are detected from the **raw** rows using the locked level-1-heading-plus-blank-neighbors rule, while the stored article text comes from the normalized rows. This is the same distinction that resolved the boundary-count discrepancies in Notebook 01.

The hard assertions below are the handoff gate. Tokenizer work does not proceed unless the shared module independently reproduces **28,472 training documents and 60 validation documents**.

In [ ]:
articles_by_split = reconstruct_development_articles(
    raw_dataset,
    normalized_development,
)

article_summary = pd.DataFrame(
    [
        {
            "split": split_name,
            "articles": len(articles),
            "expected": EXPECTED_ARTICLE_COUNTS[split_name],
            "characters": sum(len(article["text"]) for article in articles),
            "first_article_id": articles[0]["article_id"],
            "last_article_id": articles[-1]["article_id"],
        }
        for split_name, articles in articles_by_split.items()
    ]
).set_index("split")

assert article_summary.loc["train", "articles"] == 28_472
assert article_summary.loc["validation", "articles"] == 60
print("✓ Shared pipeline reproduced the audited article counts.")
article_summary


### Pause here

This chunk establishes an independently reproducible handoff from Notebook 01 to Notebook 02. A fresh runtime now reloads the pinned upstream corpus, executes the canonical shared preprocessing implementation, reruns all 17 normalization tests, and must reproduce the audited **28,472 / 60** article counts before tokenizer work begins.

**Next reviewed chunk:** choose and document the document-boundary/EOS special token, including literal-collision checks and explicit vocabulary accounting. Do not train the tokenizer yet.